In [16]:
!pip install qiskit qiskit-aer qiskit-nature qiskit-algorithms pyscf matplotlib pandas --quiet

In [17]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import transpile
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import efficient_su2

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.algorithms import GroundStateEigensolver, QEOM

from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import SLSQP, COBYLA

from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2, SamplerV2 as AerSamplerV2

HARTREE_TO_EV = 27.211386245988
EXPERIMENTAL_PI_PISTAR_EV = 7.6
np.random.seed(1)

In [18]:
def rotate_z(xyz, angle_deg):
    a = np.radians(angle_deg)
    R = np.array([[np.cos(a), -np.sin(a), 0],
                  [np.sin(a),  np.cos(a), 0],
                  [0, 0, 1]])
    return R @ np.array(xyz)

def build_geometry(twist_deg: float = 0.0) -> str:
    atoms = {k: v.copy() for k, v in equilibrium_atoms.items()}
    if twist_deg != 0.0:
        atoms["H3"] = rotate_z(atoms["H3"], twist_deg)
        atoms["H4"] = rotate_z(atoms["H4"], twist_deg)
    order = ["C1", "C2", "H1", "H2", "H3", "H4"]
    return "; ".join(f"{name[0]} {x:.6f} {y:.6f} {z:.6f}" for name in order
                      for x, y, z in [atoms[name]])

def print_geometry(label, geometry):
    print(f"{label}:")
    atoms = geometry.split("; ")
    for i, atom in enumerate(atoms):
        suffix = ";" if i < len(atoms) - 1 else ""
        print(f"  {atom}{suffix}")

# Standard planar equilibrium ethylene geometry (Angstrom)
equilibrium_atoms = {
    "C1": np.array([0.0000,  0.0000,  0.6695]),
    "C2": np.array([0.0000,  0.0000, -0.6695]),
    "H1": np.array([0.0000,  0.9289,  1.2321]),
    "H2": np.array([0.0000, -0.9289,  1.2321]),
    "H3": np.array([0.0000,  0.9289, -1.2321]),
    "H4": np.array([0.0000, -0.9289, -1.2321]),
}


geometry_equilibrium = build_geometry(0.0)
geometry_twisted = build_geometry(90.0)

# print(geometry_equilibrium)
print_geometry("Equilibrium", geometry_equilibrium)
print_geometry("\nTwisted 90", geometry_twisted)

Equilibrium:
  C 0.000000 0.000000 0.669500;
  C 0.000000 0.000000 -0.669500;
  H 0.000000 0.928900 1.232100;
  H 0.000000 -0.928900 1.232100;
  H 0.000000 0.928900 -1.232100;
  H 0.000000 -0.928900 -1.232100

Twisted 90:
  C 0.000000 0.000000 0.669500;
  C 0.000000 0.000000 -0.669500;
  H 0.000000 0.928900 1.232100;
  H 0.000000 -0.928900 1.232100;
  H -0.928900 0.000000 -1.232100;
  H 0.928900 -0.000000 -1.232100


In [19]:
driver = PySCFDriver(atom=geometry_equilibrium, basis="sto3g", charge=0, spin=0)
full_eq = driver.run()
print(f"Full STO-3G problem: {full_eq.num_spatial_orbitals} spatial orbitals, {full_eq.num_particles} particles")
print(f"Restricted Hartree-Fock (RHF) reference energy: {full_eq.reference_energy:.6f} Ha")
print(f"Nuclear repulsion energy: {full_eq.hamiltonian.nuclear_repulsion_energy:.6f} Ha")

Full STO-3G problem: 14 spatial orbitals, (8, 8) particles
Restricted Hartree-Fock (RHF) reference energy: -77.072088 Ha
Nuclear repulsion energy: 33.265090 Ha


In [20]:
# Active space: 2 electrons in 2 spatial orbitals (the pi / pi* frontier orbitals).

def ActiveSpaceTransformation(full_problem, num_active_electrons, num_spatial_orbitals):
  as_transformer = ActiveSpaceTransformer(num_electrons=num_active_electrons, num_spatial_orbitals=num_spatial_orbitals)
  as_eq = as_transformer.transform(full_problem)
  print(f"Active-space problem: {as_eq.num_spatial_orbitals} spatial orbitals, "
        f"{as_eq.num_particles} particles")
  print(f"Energy offset absorbed into the classical core (frozen-core + nuclear repulsion): {sum(as_eq.hamiltonian.constants.values()):.6f} Ha")
  return as_eq

# 2 electrons/2 orbitals
active_space_eq = ActiveSpaceTransformation(full_eq, 2, 2)


Active-space problem: 2 spatial orbitals, (1, 1) particles
Energy offset absorbed into the classical core (frozen-core + nuclear repulsion): -75.918420 Ha


In [21]:
# Qubit mapping: Jordan-Wigner vs. Parity mapping + two-qubit tapering
hamiltonian_op = active_space_eq.hamiltonian.second_q_op()

jw_mapper = JordanWignerMapper()
qubit_op_jw = jw_mapper.map(hamiltonian_op)

parity_mapper = ParityMapper(num_particles=active_space_eq.num_particles)
qubit_op_parity = parity_mapper.map(hamiltonian_op)

print(f"Jordan-Wigner mapping : {qubit_op_jw.num_qubits} qubits")
print(f"Parity mapping with tapering (2-qubit reduction) : {qubit_op_parity.num_qubits} qubits")

mapper = jw_mapper

Jordan-Wigner mapping : 4 qubits
Parity mapping with tapering (2-qubit reduction) : 2 qubits
